# Ch01-08: 데이터 품질 프레임워크 — 모범답안


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

---
## 문제 1 풀이

드리프트 유형별 검정 비교

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

def psi(ref, new, bins=10):
    bp = np.percentile(ref, np.linspace(0,100,bins+1)); bp[0]=-np.inf; bp[-1]=np.inf
    rc = np.clip(np.histogram(ref,bp)[0]/len(ref),1e-6,None)
    nc = np.clip(np.histogram(new,bp)[0]/len(new),1e-6,None)
    return np.sum((nc-rc)*np.log(nc/rc))

ref = np.random.normal(50,10,5000)
drifts = {
    '평균+2': np.random.normal(52,10,5000),
    '평균+5': np.random.normal(55,10,5000),
    '분산x1.5': np.random.normal(50,15,5000),
    '분산x2': np.random.normal(50,20,5000),
    '혼합': np.concatenate([np.random.normal(50,10,4000), np.random.normal(70,5,1000)]),
    '균일': np.random.uniform(20,80,5000),
}

results = []
for name, new in drifts.items():
    p = psi(ref, new)
    ks_s, ks_p = stats.ks_2samp(ref, new)
    bp = np.percentile(ref, np.linspace(0,100,11)); bp[0]=-np.inf; bp[-1]=np.inf
    chi_s, chi_p = stats.chisquare(np.histogram(new,bp)[0], np.histogram(ref,bp)[0])
    results.append({'Drift':name, 'PSI':p, 'KS':ks_s, 'KS_p':ks_p, 'Chi2':chi_s, 'Chi2_p':chi_p})

import pandas as pd
print(pd.DataFrame(results).to_string(index=False, float_format='%.4f'))


**결과 해석**: PSI는 평균 이동에 민감, KS는 모든 유형에 높은 검정력, χ²는 분포 형태 변화에 유용. 다각적 검정이 권장.

---
## 문제 2 풀이

스키마 검증 클래스

In [ ]:
import numpy as np, pandas as pd

class SchemaValidator:
    def __init__(self):
        self.rules = []
    def add_type(self, col, dtype): self.rules.append(('type', col, dtype))
    def add_range(self, col, lo, hi): self.rules.append(('range', col, lo, hi))
    def add_unique(self, col): self.rules.append(('unique', col))
    def add_not_null(self, col): self.rules.append(('not_null', col))
    def validate(self, df):
        results = []
        for rule in self.rules:
            if rule[0]=='type':
                ok = df[rule[1]].dtype.kind in rule[2]
                results.append(f"Type({rule[1]}): {'PASS' if ok else 'FAIL'}")
            elif rule[0]=='range':
                lo_ok = df[rule[1]].min() >= rule[2]; hi_ok = df[rule[1]].max() <= rule[3]
                results.append(f"Range({rule[1]}): {'PASS' if lo_ok and hi_ok else 'FAIL'} [{df[rule[1]].min():.1f},{df[rule[1]].max():.1f}]")
            elif rule[0]=='unique':
                ok = df[rule[1]].nunique() == len(df)
                results.append(f"Unique({rule[1]}): {'PASS' if ok else 'FAIL'}")
            elif rule[0]=='not_null':
                nn = df[rule[1]].isnull().sum()
                results.append(f"NotNull({rule[1]}): {'PASS' if nn==0 else f'FAIL ({nn} nulls)'}")
        return results

sv = SchemaValidator()
sv.add_type('age', 'if'); sv.add_range('age', 0, 150)
sv.add_not_null('name'); sv.add_unique('id')

df_test = pd.DataFrame({'id':[1,2,3,3], 'name':['a','b',None,'d'], 'age':[25,200,30,-5]})
for r in sv.validate(df_test): print(r)


**결과 해석**: 선언적 스키마 검증으로 데이터 품질 문제를 자동 탐지. 파이프라인에 통합하여 조기 경보 가능.

---
## 문제 3 풀이

시계열 데이터 프로파일러

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

np.random.seed(42)
# 시뮬레이션: 정상 + 이상 구간
n = 1000
data = np.random.normal(50, 10, n)
data[400:450] += 30  # 평균 이동
data[700:750] *= 3   # 분산 폭증

class DataProfiler:
    def __init__(self, window=50, stride=25):
        self.window, self.stride = window, stride
    def profile(self, data):
        stats_list = []
        for i in range(0, len(data)-self.window+1, self.stride):
            w = data[i:i+self.window]
            stats_list.append({'start':i, 'mean':w.mean(), 'std':w.std(), 'skew':float(pd.Series(w).skew()),
                              'null_rate':np.isnan(w).mean(), 'min':w.min(), 'max':w.max()})
        return pd.DataFrame(stats_list)
    def detect_anomalies(self, stats_df, z_thresh=2.5):
        alerts = []
        for col in ['mean','std']:
            mu, sig = stats_df[col].median(), stats_df[col].std()
            anomalous = np.abs(stats_df[col]-mu)/sig > z_thresh
            for idx in stats_df[anomalous].index:
                alerts.append(f"Window {stats_df.loc[idx,'start']}: {col} 이상 ({stats_df.loc[idx,col]:.1f})")
        return alerts

dp = DataProfiler(50, 25)
st = dp.profile(data)
alerts = dp.detect_anomalies(st)
print("탐지된 이상:")
for a in alerts: print(f"  {a}")

fig,axes = plt.subplots(3,1,figsize=(12,8),sharex=True)
axes[0].plot(data,lw=0.5); axes[0].set_title('원본')
axes[1].plot(st['start'],st['mean'],'b-o',ms=3); axes[1].set_title('윈도우 평균')
axes[2].plot(st['start'],st['std'],'r-o',ms=3); axes[2].set_title('윈도우 표준편차')
plt.tight_layout(); plt.show()


**결과 해석**: 윈도우 기반 프로파일링으로 평균 이동(구간 400-450)과 분산 폭증(구간 700-750)을 자동 탐지.

---
## 문제 4 풀이

MMD 기반 다변량 드리프트 탐지

In [ ]:
import numpy as np
from scipy.spatial.distance import cdist

np.random.seed(42)

def mmd_rbf(X, Y, gamma=None):
    if gamma is None:
        gamma = 1.0/X.shape[1]
    Kxx = np.exp(-gamma*cdist(X,X,'sqeuclidean'))
    Kyy = np.exp(-gamma*cdist(Y,Y,'sqeuclidean'))
    Kxy = np.exp(-gamma*cdist(X,Y,'sqeuclidean'))
    n,m = len(X),len(Y)
    np.fill_diagonal(Kxx, 0); np.fill_diagonal(Kyy, 0)
    return Kxx.sum()/(n*(n-1)) + Kyy.sum()/(m*(m-1)) - 2*Kxy.sum()/(n*m)

def mmd_test(X_ref, X_new, n_perm=500):
    mmd_obs = mmd_rbf(X_ref, X_new)
    combined = np.vstack([X_ref, X_new])
    n = len(X_ref); m = len(X_new)
    mmd_perms = []
    for _ in range(n_perm):
        perm = np.random.permutation(n+m)
        mmd_perms.append(mmd_rbf(combined[perm[:n]], combined[perm[n:]]))
    p_value = np.mean(np.array(mmd_perms)>=mmd_obs)
    return {'mmd': mmd_obs, 'p_value': p_value, 'permutation_mmds': mmd_perms}

# 테스트
X_ref = np.random.multivariate_normal([0,0,0], np.eye(3), 200)
X_same = np.random.multivariate_normal([0,0,0], np.eye(3), 200)
X_drift = np.random.multivariate_normal([0.5,0.3,0], 1.2*np.eye(3), 200)

r1 = mmd_test(X_ref, X_same)
r2 = mmd_test(X_ref, X_drift)
print(f"Same dist: MMD={r1['mmd']:.6f}, p={r1['p_value']:.4f}")
print(f"Drifted:   MMD={r2['mmd']:.6f}, p={r2['p_value']:.4f}")


**결과 해석**: MMD는 커널 기반으로 고차원 분포 차이를 비모수적으로 탐지. 순열 검정으로 통계적 유의성 평가. 같은 분포에서는 p>0.05, 드리프트 시 p<0.05.

---
## 확장 토론

### 데이터 품질 모니터링 체크리스트

1. 스키마: 타입, 범위, null, 유일성
2. 분포: PSI, KS (단변량), MMD (다변량)
3. 프로파일: 윈도우 통계량 모니터링
4. 자동화: 임계값 기반 알림 파이프라인